# Lab 2 - Comitês (Ensembles)

Classificadores: **Random Forest**, **AdaBoost** e **XGBoost**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import os
from datetime import datetime

## Carregamento dos Dados Processados

In [ ]:
data_dir = './dados_processados'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').values.ravel()
X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').values.ravel()

mod_time = datetime.fromtimestamp(os.path.getmtime(f'{data_dir}/X_train.parquet'))
print(f'Dados carregados (gerados em {mod_time.strftime("%Y-%m-%d %H:%M")})')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}  |  y_test: {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')
print(f'  Churn rate treino: {y_train.mean():.3f}  |  teste: {y_test.mean():.3f}')

## Configuração do MLflow e Função de Avaliação

O MLflow registra automaticamente parâmetros, métricas e artefatos de cada modelo.
Todos os notebooks usam o mesmo experimento (`Lab2-Churn`), então os runs aparecem juntos na UI.

**Para visualizar os resultados após rodar os notebooks:**
```bash
cd notebooks
mlflow ui
# Abrir http://127.0.0.1:5000 no navegador
```

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

mlflow.set_experiment("Lab2-Churn")


def avaliar_modelo(nome, y_true, y_pred):
    """Avalia um modelo, printa o relatório e retorna dict com métricas."""
    metricas = {
        'Modelo': nome,
        'Acuracia': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    print(f'\n=== {nome} ===')
    print(classification_report(y_true, y_pred, target_names=['Não cancelou', 'Cancelou']))
    print(f'Kappa: {metricas["Kappa"]:.4f}')
    return metricas


def logar_mlflow(metricas, params, artefatos=None):
    """Registra métricas, parâmetros e artefatos no run MLflow ativo."""
    for k, v in params.items():
        mlflow.log_param(k, v)
    for k, v in metricas.items():
        if k != 'Modelo':
            mlflow.log_metric(k.lower().replace('-', '_'), v)
    if artefatos:
        for caminho in artefatos:
            mlflow.log_artifact(caminho)

## Nota sobre Normalização

Modelos baseados em árvores (Random Forest, AdaBoost, XGBoost) **não precisam** de normalização.
Usamos `X_train` e `X_test` diretamente.

## Exemplo de Uso — Padrão MLflow

Comitês não precisam de scaler, então usamos `X_train`/`X_test` diretamente.
Além das métricas, logamos o gráfico de feature importance como artefato.

In [ ]:
# ─── Exemplo: Random Forest ─────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="RandomForest"):
    # 1. Parâmetros
    n_est = 200
    params = {'modelo': 'RandomForest', 'n_estimators': n_est, 'scaler': 'nenhum'}

    # 2. Treinar (sem scaler — árvores não precisam)
    model = RandomForestClassifier(n_estimators=n_est, random_state=42)
    model.fit(X_train, y_train)

    # 3. Predizer e avaliar
    y_pred = model.predict(X_test)
    metricas = avaliar_modelo('RandomForest', y_test, y_pred)

    # 4. Logar no MLflow (com feature importance como artefato)
    fig, ax = plt.subplots(figsize=(10, 6))
    importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values()
    importances.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Feature Importance — Random Forest')
    plt.tight_layout()
    fi_path = '../relatorio/imagens/2c_com_feature_importance_rf.png'
    plt.savefig(fi_path, dpi=150, bbox_inches='tight')
    plt.show()

    logar_mlflow(metricas, params, artefatos=[fi_path])
    mlflow.sklearn.log_model(model, 'modelo_rf')